In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import os
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import log_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [4]:
train_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
def clean_prompt(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.replace("based on the given context", "")
    text = text.replace("carefully", "")
    text = text.replace("among the listed options", "")
    text = text.replace("from the following choices", "")
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["group"] = train_df["prompt"].apply(clean_prompt)
test_df["group"] = test_df["prompt"].apply(clean_prompt)


In [6]:
def make_pairwise(df, train=True):

    rows = []

    for _, row in df.iterrows():

        for option in ["A", "B", "C", "D", "E"]:
            option_text = row[option]
            if pd.isna(option_text):
                continue

            rows.append({
                "id": row["id"],
                "group": row["group"],
                "question": row["prompt"],
                "option": option_text,
                "option_name": option,
                "text": (
                    "question: "
                    + row["prompt"]
                    + " [SEP] option: "
                    + option_text
                ),
                "label": int(option == row["answer"]) if train else -1,
            })

    return pd.DataFrame(rows)


pair_train = make_pairwise(train_df)
pair_test = make_pairwise(test_df, train=False)

print(pair_train.shape)
pair_train.head()


(10000, 7)


,id,group,question,option,option_name,text,label
0,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,question: Pick the best possible answer: What ...,0
1,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,question: Pick the best possible answer: What ...,1
2,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,question: Pick the best possible answer: What ...,0
3,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,question: Pick the best possible answer: What ...,0
4,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,question: Pick the best possible answer: What ...,0


In [7]:
from sklearn.model_selection import GroupShuffleSplit

# Keep a slightly richer vocabulary and add a small character-level safeguard for rare words.
max_features = 40000
max_len = 256

tokenizer = Tokenizer(num_words=max_features, oov_token="<UNK>")
tokenizer.fit_on_texts(pair_train["text"])


In [8]:
X = tokenizer.texts_to_sequences(pair_train["text"])
X = pad_sequences(X, maxlen=max_len, padding="post", truncating="post")

y = pair_train["label"].values
groups = pair_train["group"].values

# Use a grouped split so validation is not leakage-prone.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(gss.split(X, y, groups))

X_train = X[train_idx]
X_valid = X[valid_idx]

y_train = y[train_idx]
y_valid = y[valid_idx]

print("Train size:", X_train.shape[0])
print("Valid size:", X_valid.shape[0])


Train size: 8015
Valid size: 1985


In [9]:
class MCQDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(
    MCQDataset(X_train, y_train),
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

valid_loader = DataLoader(
    MCQDataset(X_valid, y_valid),
    batch_size=256,
    shuffle=False,
    num_workers=0,
)


In [10]:
class BiLSTMAttention(nn.Module):

    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )

        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(0.35)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)

        out, _ = self.lstm(x)

        attn_scores = self.attention(out).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights.unsqueeze(-1) * out, dim=1)

        context = self.dropout(context)
        return self.fc(context).squeeze(1)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = BiLSTMAttention(max_features).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)


Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [12]:
epochs = 8

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}  Loss: {train_loss/len(train_loader):.4f}")


Epoch 1/8  Loss: 0.5034
Epoch 2/8  Loss: 0.3393
Epoch 3/8  Loss: 0.1415
Epoch 4/8  Loss: 0.0771
Epoch 5/8  Loss: 0.0515
Epoch 6/8  Loss: 0.0311
Epoch 7/8  Loss: 0.0193
Epoch 8/8  Loss: 0.0237


In [13]:
model.eval()

valid_scores = []

with torch.no_grad():

    for X_batch, _ in valid_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.sigmoid(outputs)

        valid_scores.extend(
            probs.cpu().numpy()
        )

valid_scores = np.array(valid_scores)

print(valid_scores[:10])

[1.7490186e-04 9.9952257e-01 1.6091533e-04 1.4576496e-04 1.8060328e-04
 9.9773204e-01 1.4607105e-03 1.4453599e-03 1.4580599e-04 1.4612429e-04]


In [14]:
valid_df = pair_train.iloc[valid_idx].copy()
valid_df = valid_df.reset_index(drop=True)
valid_df["score"] = valid_scores

print(valid_df[["id", "option_name", "label", "score"]].head(10))
print(valid_df.shape)


   id option_name  label     score
0   4           A      0  0.000175
1   4           B      1  0.999523
2   4           C      0  0.000161
3   4           D      0  0.000146
4   4           E      0  0.000181
5   5           A      1  0.997732
6   5           B      0  0.001461
7   5           C      0  0.001445
8   5           D      0  0.000146
9   5           E      0  0.000146
(1985, 8)


In [15]:
predictions = []
true_answers = []

for _, group in valid_df.groupby("id"):
    group = group.sort_values("score", ascending=False)
    predictions.append(group["option_name"].tolist()[:3])
    true_answers.append(group.loc[group["label"] == 1, "option_name"].values[0])

print(predictions[:5])
print(true_answers[:5])


[['B', 'E', 'A'], ['A', 'B', 'C'], ['A', 'B', 'E'], ['D', 'C', 'B'], ['E', 'B', 'C']]
['B', 'A', 'A', 'D', 'E']


In [16]:
def map3(y_true, y_pred):
    score = 0.0
    for true, pred in zip(y_true, y_pred):
        if true == pred[0]:
            score += 1.0
        elif true == pred[1]:
            score += 0.5
        elif true == pred[2]:
            score += 1.0 / 3.0
    return score / len(y_true)

map_score = map3(true_answers, predictions)
print("Validation MAP@3:", round(map_score, 5))


Validation MAP@3: 0.9937


In [17]:
top1 = [p[0] for p in predictions]
acc = np.mean(np.array(top1) == np.array(true_answers))
print("Validation Accuracy:", round(acc, 5))


Validation Accuracy: 0.98741


In [18]:
test_sequences = tokenizer.texts_to_sequences(pair_test["text"])
test_sequences = pad_sequences(test_sequences, maxlen=max_len, padding="post", truncating="post")

test_loader = DataLoader(
    torch.LongTensor(test_sequences),
    batch_size=256,
    shuffle=False,
)


In [19]:
model.eval()

test_scores = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        test_scores.extend(probs.cpu().numpy())

pair_test["score"] = test_scores

predictions = []
ids = []

for qid, group in pair_test.groupby("id"):
    group = group.sort_values("score", ascending=False)
    ids.append(qid)
    predictions.append(" ".join(group["option_name"].tolist()[:3]))


In [20]:
submission = pd.DataFrame({

    "ID": ids,

    "Prediction": predictions

})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

print("submission.csv saved")

   ID Prediction
0   1      A B C
1   2      B C E
2   3      B A E
3   4      E C D
4   5      C A D
submission.csv saved
